# Exercise 5 · Test or application: which is at fault?

A test fails after a teammate's change. You decide what to repair: the test or the page.

**Goal:** Read a failing browser test, decide if the test or the page is wrong, and repair that one.

**Why it matters:** A test changed to pass can hide a fault that every user sees.

**Before you start:**
- You did exercise 3 of today. Its Step 3 installed the browser for tests.
- You installed Codex and signed in (course README, step 1).

**What the check measures:** `test_page.py` uses the new label and still expects 2 rows. Your last run passed. You committed both repairs. Your two verdicts are right.

**Done when:** the last cell prints `Done:`.

## The situation

`cases.html` is a case list with a Status filter. `test_page.py` chooses Open and expects 2 rows. A teammate changes the page twice. The files hold made-up data.

## Step 1 · Write the practice files

Run the cell. It writes the page, the test and the teammate's two changes into `practice/`, beside this notebook. It never overwrites a file.

Expect five lines that start with `Wrote practice/`, or `Kept` for a file that is already there.

In [ ]:
FILES = {
    "cases.html": """<!doctype html>
<html><head><meta charset="utf-8"><title>Case list</title></head>
<body>
<label for="status">Status</label>
<select id="status"><option value="">All</option><option value="open">Open</option><option value="closed">Closed</option></select>
<table><tbody id="cases"></tbody></table>
<script>
const CASES = [{id: "C-1", status: "open"}, {id: "C-2", status: "open"}, {id: "C-3", status: "closed"}];
function show(status) {
  const body = document.getElementById("cases");
  clearTimeout(window.pending);
  body.innerHTML = "";
  window.pending = setTimeout(() => {
    for (const c of CASES.filter(c => !status || c.status === status)) {
      const row = document.createElement("tr");
      row.dataset.testid = "case-row";
      row.innerHTML = `<td>${c.id}</td><td>${c.status === "open" ? "Open" : "Closed"}</td>`;
      body.appendChild(row);
    }
  }, 300);
}
document.getElementById("status").addEventListener("change", e => show(e.target.value));
show("");
</script></body></html>
""",
    "test_page.py": """from pathlib import Path
from playwright.sync_api import Page, expect

PAGE = Path(__file__).parent / "cases.html"


def test_filter_shows_only_open_cases(page: Page) -> None:
    page.goto(PAGE.as_uri())
    page.get_by_label("Status", exact=True).select_option("open")
    rows = page.get_by_test_id("case-row")
    expect(rows).to_have_count(2)
    expect(rows.filter(has_text="Closed")).to_have_count(0)
""",
    ".gitignore": "__pycache__/\n.pytest_cache/\ntest-results/\nresults.xml\n",
}
# The teammate's two changes: copies of cases.html, each with a note at the top
NOTE = "<body>\n<!-- Change {}, from a teammate: {} -->"
renamed = FILES["cases.html"].replace(">Status<", ">Case status<")
FILES["changes/change-a.html"] = renamed.replace("<body>", NOTE.format(
    "A", 'the label now reads "Case status", as on the new paper form. The rename is meant.'))
FILES["changes/change-b.html"] = renamed.replace("<body>", NOTE.format(
    "B", "the filter rule moved into its own function, matches(). Users should see no change.")).replace(
    "function show(", "function matches(c, status) {\n  return !status || c.status === status;\n}\nfunction show(").replace(
    "c => !status || c.status === status", "c => matches(c)")

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Commit the start, then run the test

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-05-test-or-application/practice
git init -q -b main && git add -A && git commit -qm "Start"
uv run --with pytest-playwright==0.10.0 pytest test_page.py --tracing=retain-on-failure --junitxml=results.xml
```

The last line is "the Step 2 run". It runs the test headless: no window opens. Expect `1 passed`.

## Step 3 · Change A: read the failure

```sh
cp changes/change-a.html cases.html
```

Do the Step 2 run. Expect `1 failed` after about 30 seconds. Read the lines that start with `E`:

- `TimeoutError: Locator.select_option: Timeout 30000ms exceeded.`
- `- waiting for get_by_label("Status", exact=True)`

Read the teammate's note at the top of `cases.html`. Optional: replay the test with `uv run --with pytest-playwright==0.10.0 playwright show-trace test-results/*/trace.zip`.

### Learn: a timeout means a check never passed

"If the required checks do not pass within the given timeout, action fails with the TimeoutError." ([Auto-waiting](https://playwright.dev/python/docs/actionability)). `exact=True` matches only a whole label, and no label is "Status" now.

## Step 4 · Change A: decide, then repair

Is the test at fault, or the page (the application)? Write `test` or `application` for `CHANGE_A` in the cell after Step 6, then run it.

<details><summary>Show the answer</summary>

The test. Why: the rename is meant, and the page still works. The test looks for the old label.

</details>

In `test_page.py`, change `"Status"` to `"Case status"`. Change nothing else. Do the Step 2 run. Expect `1 passed`. Commit:

```sh
git commit -qam "Find the status control by its new label"
```

## Step 5 · Change B: read the failure, decide

```sh
cp changes/change-b.html cases.html
```

Do the Step 2 run. Expect `1 failed`, with `Locator expected to have count '2'` and `Actual value: 3`. Write your verdict for `CHANGE_B` in the same cell, and run it again.

<details><summary>Show the answer</summary>

The application. Why: there are 2 open cases. A user sees a closed case under Open. The test did its job.

</details>

## Step 6 · Codex repairs the page, you check it

Refuse three repairs: an expected count of 3, a longer wait, a skip. Each turns the test green and hides the fault. Start Codex with `codex --no-daemon`. If it asks "Trust this folder?", choose `1. Trust and continue`. Type this request and press Enter:

```text
test_page.py fails after change B: the Open filter shows 3 rows, not 2. Find the cause in cases.html and fix it. Do not change test_page.py. I will run the test myself.
```

Codex says it finished. Read its change with `/diff` (press `q` to close), then type `/exit`. Do not take its word: do the Step 2 run. Keep the repair only if the test passes. Commit with the cause:

```sh
git commit -qam "Pass the status to matches(): Open showed every case"
```

In [ ]:
CHANGE_A = ""  # test or application
CHANGE_B = ""  # test or application
print(f"Saved your verdicts: change A = {CHANGE_A!r}, change B = {CHANGE_B!r}")

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again. Set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect five `ok:` lines, then `Done:`.

In [ ]:
# Check your work: test_page.py, results.xml, Git, verdicts
import re
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")


def git(*args) -> str:
    return subprocess.run(["git", "-C", str(PRACTICE), *args], capture_output=True, text=True).stdout


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    test, page, results = (PRACTICE / name for name in ("test_page.py", "cases.html", "results.xml"))
    source = test.read_text(encoding="utf-8") if test.is_file() else ""
    if not re.search(r"""get_by_label\(\s*["']Case status["']""", source):
        return 'test_page.py does not look for the label "Case status". Update the locator (Step 4).'
    print('ok: test_page.py finds the label "Case status"')
    if (any(word in source for word in ("timeout", "sleep", "skip")) or re.search(r"to_have_count\(\s*[13-9]", source)
            or not re.search(r"to_have_count\(\s*2\s*\)", source)):
        return "test_page.py has a new count, a wait or a skip. Undo it, and repair the page (Step 6)."
    print("ok: test_page.py still expects 2 rows, with no wait and no skip")
    if not results.is_file():
        return "practice/results.xml is missing. Do the Step 2 run."
    suites = list(ET.parse(results).getroot().iter("testsuite"))
    tests = sum(int(s.get("tests", 0)) for s in suites)
    bad = sum(int(s.get(key, 0)) for s in suites for key in ("failures", "errors", "skipped"))
    if tests < 1 or bad:
        return "your last run did not pass. Repair the page (Step 6)."
    if results.stat().st_mtime < max(test.stat().st_mtime, page.stat().st_mtime):
        return "you changed a file after your last run. Do the Step 2 run again."
    print("ok: your last run passed, after your last change")
    if (not (PRACTICE / ".git").is_dir() or len(git("log", "--format=%s").splitlines()) < 3
            or git("status", "--porcelain", "cases.html", "test_page.py")):
        return "a repair is not committed. Commit each one (Steps 4 and 6)."
    print("ok: both repairs are committed")
    verdicts = [str(globals().get(name, "")).strip().lower() for name in ("CHANGE_A", "CHANGE_B")]
    if not verdicts[0].startswith("test") or not verdicts[1].startswith("app"):
        return "a verdict is empty or not right. Fill the cell after Step 6 and run it."
    print("ok: your verdicts are right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Do the Step 2 run again."
print(f"Not yet: {problem}" if problem else
      "Done: you repaired the test for change A and the page for change B.")

## What you learned

- Read the failure first. The renamed label was the test's fault. The wrong count was the page's.
- When the page is at fault, keep the test. Run it to see the repair work.

**Reflect:** Before today, which repair for change B looked right to you?